In [18]:
import pandas as pd

first_set = pd.read_json('../data/raw/rev2_docs_since_2020_01_01.json', lines=True)
first_set = first_set[["id", "product_id", "name", "desc", "brand", "shop_cat", "price"]]

second_set = pd.read_pickle('../data/working/dedup_preprocessed_rev2_docs_since_2020_01_01_only_de_strict.pkl.gz')
second_set = second_set[["id", "product_id", "name", "desc", "brand", "shop_cat", "price"]]

third_set = pd.read_pickle('../data/working/dedup_preprocessed_rev2_docs_since_2020_01_01_only_de_strict_only_long_name.pkl.gz')
third_set = third_set[["id", "product_id", "name", "desc", "brand", "shop_cat", "price"]]

fourth_set = pd.read_pickle('../data/working/dedup_preprocessed_rev2_docs_since_2020_01_01_only_de_strict_only_long_title_only_mainentity.pkl.gz')
fourth_set = fourth_set[["id", "product_id", "name", "desc", "brand", "shop_cat", "price"]]


In [ ]:
print("First set keys:", first_set['id'].dtype, len(first_set))
print("Second set keys:", second_set['id'].dtype, len(second_set))
print("Third set keys:", third_set['id'].dtype, len(third_set))
print("Fourth set keys:", fourth_set['id'].dtype, len(fourth_set))

# 🧩 Choose a column that uniquely identifies each record
# Replace 'id' with the actual column name if it's different
key_col = 'id' if 'id' in first_set.columns else first_set.columns[0]

# 1️⃣ In first but not in second
missing_1_vs_2 = first_set.loc[~first_set[key_col].isin(second_set[key_col])]

# 2️⃣ In second but not in third
missing_2_vs_3 = second_set.loc[
    ~second_set[key_col].isin(third_set[key_col])
]

# 3️⃣ In third but not in fourth
missing_3_vs_4 = third_set.loc[
    ~third_set[key_col].isin(fourth_set[key_col])
]

print("removed_1_vs_2:", len(missing_1_vs_2))
print("missing_2_vs_3:", len(missing_2_vs_3))
print("missing_3_vs_4:", len(missing_3_vs_4))


# 🧪 Downsample each to max 100 rows (random but reproducible)
missing_1_vs_2_sample = missing_1_vs_2.sample(n=min(100, len(missing_1_vs_2)), random_state=42)
missing_2_vs_3_sample = missing_2_vs_3.sample(n=min(100, len(missing_2_vs_3)), random_state=42)
missing_3_vs_4_sample = missing_3_vs_4.sample(n=min(100, len(missing_3_vs_4)), random_state=42)

# ✅ Optional: save for inspection
missing_1_vs_2_sample.to_csv('../data/debug/missing_first_vs_second.csv', index=False)
missing_2_vs_3_sample.to_csv('../data/debug/missing_second_vs_third.csv', index=False)
missing_3_vs_4_sample.to_csv('../data/debug/missing_third_vs_fourth.csv', index=False)

First set keys: int64 6309224
Second set keys: int64 3308479
Third set keys: int64 3030931
Fourth set keys: int64 2979790
missing_1_vs_2: 3000742
missing_2_vs_3: 277548
missing_3_vs_4: 51141
